In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver # InMemorySaver save state in RAM/Memory

In [2]:
llm = ChatOllama(model = 'llama3.2')

In [3]:
class JokeState(TypedDict):
    topic: str
    joke : str
    explanation: str

In [4]:
def generate_joke(state: JokeState):
    prompt = f'Write joke on \n {state['topic']}'

    joke = llm.invoke(prompt).content

    return {'joke': joke}

In [5]:
def generate_explanation(state: JokeState):
    prompt = f'Write a explantion for the joke \n {state['joke']}'

    explanation = llm.invoke(prompt).content

    return {'explanation' : explanation}

In [6]:
graph = StateGraph(JokeState)
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer = checkpointer)


In [7]:
config1 = {"configurable": {'thread_id':"1"}}
response = workflow.invoke({'topic': 'Indian traffic'}, config = config1)

In [8]:
response

{'topic': 'Indian traffic',
 'joke': 'Why did the Indian driver bring a ladder with him while driving?\n\nBecause he wanted to take his journey to the next level! (get it? like in traffic?)',
 'explanation': 'This joke is a play on words, using the multiple meanings of "level" to create a pun. In this case, "level" has two different interpretations:\n\n1. In a literal sense, when referring to driving or transportation, "taking his journey to the next level" means improving or advancing one\'s skills or experience while driving.\n2. Figuratively, "leveling up" is a gaming term that refers to progressing from a lower level of difficulty to a higher one.\n\nThe joke relies on this wordplay, using the phrase "take his journey to the next level" in a literal sense (i.e., physically climbing with a ladder) and simultaneously referencing the gaming concept. The humor comes from the unexpected twist on the common phrase and the clever connection between the setup and punchline.\n\nIn this spec

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Indian traffic', 'joke': 'Why did the Indian driver bring a ladder with him while driving?\n\nBecause he wanted to take his journey to the next level! (get it? like in traffic?)', 'explanation': 'This joke is a play on words, using the multiple meanings of "level" to create a pun. In this case, "level" has two different interpretations:\n\n1. In a literal sense, when referring to driving or transportation, "taking his journey to the next level" means improving or advancing one\'s skills or experience while driving.\n2. Figuratively, "leveling up" is a gaming term that refers to progressing from a lower level of difficulty to a higher one.\n\nThe joke relies on this wordplay, using the phrase "take his journey to the next level" in a literal sense (i.e., physically climbing with a ladder) and simultaneously referencing the gaming concept. The humor comes from the unexpected twist on the common phrase and the clever connection between the setup and punchli

In [20]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Indian traffic', 'joke': 'Why did the Indian driver bring a ladder to the car?\n\nBecause he wanted to take his driving to the next level... but ended up getting stuck in neutral! (get it?)', 'explanation': 'The joke relies on a play on words, specifically a pun. The phrase "take his driving to the next level" is typically used to describe improving one\'s skills or taking their career to new heights. However, in this context, the driver literally uses a ladder to elevate themselves to a higher position while driving.\n\nThe punchline subverts expectations by introducing an unexpected twist: getting stuck in neutral gear. "Neutral" has a double meaning here - it refers to both the gear in a vehicle and being at an intermediate stage (i.e., not moving forward or backward).\n\nThis joke requires some cultural knowledge of Indian culture, as drivers are often referred to as "chauffeurs." The use of this term adds to the humor, creating a lighthearted poke 

In [18]:
# Time Travel
workflow.get_state({'configurable':{'thread_id':'1', 'checkpoint_id':'1f170b8b-a052-64e6-8000-1c8e13950b13'}})

StateSnapshot(values={'topic': 'Indian traffic'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f170b8b-a052-64e6-8000-1c8e13950b13'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-06-25T17:10:12.238866+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f170b8b-a051-6398-bfff-3486ff933199'}}, tasks=(PregelTask(id='811192ec-ff63-2e86-ac3a-243b0c31faca', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the Indian driver bring a ladder with him while driving?\n\nBecause he wanted to take his journey to the next level! (get it? like in traffic?)'}),), interrupts=())

In [ ]:
# Time travel
config3 = {'configurable': {'thread_id':'1', 'checkpoint_id':'1f170b8b-a052-64e6-8000-1c8e13950b13'}}
workflow.invoke(None, config = config3)

{'topic': 'Indian traffic',
 'joke': 'Why did the Indian driver bring a ladder to the car?\n\nBecause he wanted to take his driving to the next level... but ended up getting stuck in neutral! (get it?)',
 'explanation': 'The joke relies on a play on words, specifically a pun. The phrase "take his driving to the next level" is typically used to describe improving one\'s skills or taking their career to new heights. However, in this context, the driver literally uses a ladder to elevate themselves to a higher position while driving.\n\nThe punchline subverts expectations by introducing an unexpected twist: getting stuck in neutral gear. "Neutral" has a double meaning here - it refers to both the gear in a vehicle and being at an intermediate stage (i.e., not moving forward or backward).\n\nThis joke requires some cultural knowledge of Indian culture, as drivers are often referred to as "chauffeurs." The use of this term adds to the humor, creating a lighthearted poke at stereotypes.'}

In [ ]:
# Another thread
config2 = {'configurable' : {'thread_id':2}}
response = workflow.invoke({'topic': 'india railways'}, config= config2)
response

{'topic': 'india railways',
 'joke': 'Why did the Indian Railways conductor bring a ladder to work?\n\nBecause he wanted to take his service to a whole new level! (get it?)',
 'explanation': 'This joke is a play on words, using a pun to create humor. The phrase "take it to a whole new level" has a double meaning here.\n\nIn the context of Indian Railways, "service" refers not only to the train service being provided but also to the service that the conductor provides, such as assisting passengers and ensuring their safety.\n\nThe ladder is used in this joke because, literally, one would need a ladder to reach higher levels. So when the conductor brings a ladder to work, it\'s a clever play on words to say that he wants to "take his service" (the train service) to a "whole new level". The punchline relies on the listener being familiar with the phrase "take it to a whole new level", which is often used to describe improving or upgrading something.\n\nThe joke is a lighthearted and cleve

In [13]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'india railways', 'joke': 'Why did the Indian Railways conductor bring a ladder to work?\n\nBecause he wanted to take his service to a whole new level! (get it?)', 'explanation': 'This joke is a play on words, using a pun to create humor. The phrase "take it to a whole new level" has a double meaning here.\n\nIn the context of Indian Railways, "service" refers not only to the train service being provided but also to the service that the conductor provides, such as assisting passengers and ensuring their safety.\n\nThe ladder is used in this joke because, literally, one would need a ladder to reach higher levels. So when the conductor brings a ladder to work, it\'s a clever play on words to say that he wants to "take his service" (the train service) to a "whole new level". The punchline relies on the listener being familiar with the phrase "take it to a whole new level", which is often used to describe improving or upgrading something.\n\nThe joke is a lig

In [16]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'india railways', 'joke': 'Why did the Indian Railways conductor bring a ladder to work?\n\nBecause he wanted to take his service to a whole new level! (get it?)', 'explanation': 'This joke is a play on words, using a pun to create humor. The phrase "take it to a whole new level" has a double meaning here.\n\nIn the context of Indian Railways, "service" refers not only to the train service being provided but also to the service that the conductor provides, such as assisting passengers and ensuring their safety.\n\nThe ladder is used in this joke because, literally, one would need a ladder to reach higher levels. So when the conductor brings a ladder to work, it\'s a clever play on words to say that he wants to "take his service" (the train service) to a "whole new level". The punchline relies on the listener being familiar with the phrase "take it to a whole new level", which is often used to describe improving or upgrading something.\n\nThe joke is a li